# HaMeR ViT-B — Inference Demo
This notebook loads the trained ViT-B checkpoint and runs inference on sample images.

**Requirements:** Mount your Google Drive with the HaMeR repo and the trained checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/hamer
!pip install -e .[all] -q
!pip install -e third-party/ViTPose -q
import sys
sys.path.insert(0, '/content/drive/MyDrive/hamer')
print('Setup done!')

In [ ]:
import os, torch, cv2
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from hamer.configs import get_config
from hamer.models import HAMER

CKPT_PATH   = '/content/drive/MyDrive/hamer/checkpoints_vitb_v2/best.ckpt'
CONFIG_PATH = '/content/drive/MyDrive/hamer/_DATA/hamer_ckpts_vitb/model_config.yaml'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model_cfg = get_config(CONFIG_PATH, update_cachedir=True)
model_cfg.defrost()
if 'BBOX_SHAPE' not in model_cfg.MODEL:
    model_cfg.MODEL.BBOX_SHAPE = [192, 256]
if 'PRETRAINED_WEIGHTS' in model_cfg.MODEL.BACKBONE:
    model_cfg.MODEL.BACKBONE.pop('PRETRAINED_WEIGHTS')
model_cfg.freeze()

model = HAMER(cfg=model_cfg)
ckpt = torch.load(CKPT_PATH, map_location='cpu')
state_dict = {k.replace('model.', ''): v for k, v in ckpt['model_state_dict'].items()}
model.load_state_dict(state_dict, strict=False)
model = model.to(device)
model.eval()
print(f'Model loaded! Backbone: {model_cfg.MODEL.BACKBONE.TYPE}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'pos_embed shape: {model.backbone.pos_embed.shape}  (should be [1, 193, 768] for ViT-B)')

In [ ]:
# Hand detector (ViTDet — unchanged from original HaMeR)
import hamer
from pathlib import Path
from detectron2.config import LazyConfig
from hamer.utils.utils_detectron2 import DefaultPredictor_Lazy

cfg_path = Path(hamer.__file__).parent/'configs'/'cascade_mask_rcnn_vitdet_h_75ep.py'
detectron2_cfg = LazyConfig.load(str(cfg_path))
detectron2_cfg.train.init_checkpoint = 'https://dl.fbaipublicfiles.com/detectron2/ViTDet/COCO/cascade_mask_rcnn_vitdet_h/f328730692/model_final_f05665.pkl'
for i in range(3):
    detectron2_cfg.model.roi_heads.box_predictors[i].test_score_thresh = 0.25
detector = DefaultPredictor_Lazy(detectron2_cfg)
print('Detector loaded!')

In [ ]:
from hamer.datasets.vitdet_dataset import ViTDetDataset
from hamer.utils import recursive_to
from hamer.utils.renderer import Renderer

def run_inference(img_path):
    img_cv2 = cv2.imread(img_path)
    img_h, img_w = img_cv2.shape[:2]

    # Detect hands
    det_out = detector(img_cv2)
    det_instances = det_out['instances']
    valid_idx = (det_instances.pred_classes == 0) & (det_instances.scores > 0.5)
    boxes = det_instances.pred_boxes.tensor[valid_idx].cpu().numpy()
    right = det_instances.pred_classes[valid_idx].cpu().numpy()
    print(f'Hands detected: {len(boxes)}')

    if len(boxes) == 0:
        print('No hands detected!')
        return

    # Run HaMeR ViT-B
    dataset = ViTDetDataset(model_cfg, img_cv2, boxes, right, rescale_factor=2.0)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=False)

    all_verts, all_cam_t = [], []
    with torch.no_grad():
        for batch in dataloader:
            batch = recursive_to(batch, device)
            out = model(batch)
            all_verts.append(out['pred_vertices'].cpu())
            all_cam_t.append(out['pred_cam_t'].cpu())

    all_verts = torch.cat(all_verts)
    all_cam_t = torch.cat(all_cam_t)

    # Render
    renderer = Renderer(model_cfg, faces=model.mano.faces)
    focal_length = model_cfg.EXTRA.FOCAL_LENGTH
    img_rgb = img_cv2[:, :, ::-1] / 255.0

    cam_view = renderer.render_rgba_multiple(
        all_verts.numpy(), cam_t=all_cam_t.numpy(),
        render_res=[img_w, img_h],
        mesh_base_color=(0.6, 0.8, 1.0),
        scene_bg_color=(1, 1, 1),
        focal_length=focal_length,
    )
    valid_mask = cam_view[:, :, 3:4] > 0
    img_out = img_rgb * (1 - valid_mask) + cam_view[:, :, :3] * valid_mask

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(img_rgb)
    axes[0].set_title('Input', fontsize=14)
    axes[0].axis('off')
    axes[1].imshow(img_out)
    axes[1].set_title('HaMeR ViT-B reconstruction', fontsize=14)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

# Run on example images
for img in ['example_data/test1.jpg', 'example_data/test5.jpg']:
    print(f'\n--- {img} ---')
    run_inference(img)